# Error Case 27 — 500/503: PostgreSQL Max Connections Exceeded

**Error:** `500 Internal Server Error` / `503 Service Unavailable`  
**Root cause:** All PostgreSQL connections consumed — new connection attempts fail  
**Fix:** Reduce pool size, add PgBouncer, or increase PostgreSQL max_connections

## Flow
```
Polaris pool: max-size=300 (your config)
PostgreSQL: max_connections=100 (default)
  → Polaris tries to open 300 connections
  → PostgreSQL: "sorry, too many clients already"
  → Agroal: cannot create connection → pool exhausted
  → Requests: 500 "Unable to acquire connection"
```

## Your current config risk
```
Polaris values.yaml:
  QUARKUS_DATASOURCE_JDBC_MAX_SIZE: 300

PostgreSQL default max_connections: 100
  → Polaris alone exceeds PostgreSQL limit
  → If other services share PostgreSQL → worse
  → Must match pool size to PostgreSQL capacity
```

In [ ]:
import sys
sys.path.insert(0, '..')
import importlib, polaris_test_utils
importlib.reload(polaris_test_utils)
from polaris_test_utils import *
import subprocess

cleanup()
print('✅ Setup complete')

In [ ]:
# Cell 2: Check current PostgreSQL max_connections and active connections
def pg_query(sql):
    """Run query via pgpool → primary."""
    # Find primary pod
    for i in range(3):
        pod = f'benchmarks-postgresql-postgresql-ha-postgresql-{i}'
        result = subprocess.run([
            'kubectl', 'exec', '-n', 'datahub-hynix', pod,
            '--', 'env', 'PGPASSWORD=polaris',
            'psql', '-U', 'polaris', '-d', 'polaris',
            '-c', 'SELECT pg_is_in_recovery();'
        ], capture_output=True, text=True)
        if 'f' in result.stdout:
            r = subprocess.run([
                'kubectl', 'exec', '-n', 'datahub-hynix', pod,
                '--', 'env', 'PGPASSWORD=polaris',
                'psql', '-U', 'polaris', '-d', 'polaris', '-c', sql
            ], capture_output=True, text=True)
            return r.stdout
    return 'Primary not found'

print('=== PostgreSQL connection stats ===')
print(pg_query('SHOW max_connections;'))
print(pg_query("""
SELECT count(*) as active_connections,
       max_conn.setting::int as max_connections,
       max_conn.setting::int - count(*) as available
FROM pg_stat_activity,
     pg_settings max_conn
WHERE max_conn.name = 'max_connections'
GROUP BY max_conn.setting;
"""))

print('=== Connections by application ===')
print(pg_query("""
SELECT application_name, count(*) as connections
FROM pg_stat_activity
WHERE state IS NOT NULL
GROUP BY application_name
ORDER BY connections DESC;
"""))

In [ ]:
# Cell 3: Simulate connection exhaustion
# Temporarily reduce max_connections to simulate exhaustion
import threading, time

print('Simulating connection pool pressure...')
print('Sending concurrent requests to exhaust pool...')

errors = []
success = []
lock = threading.Lock()

def hammer_request(req_id):
    try:
        r = requests.get(
            f'{BASE_MGMT}/catalogs',
            headers=h(),
            timeout=10
        )
        with lock:
            if r.status_code == 200:
                success.append(req_id)
            else:
                errors.append({'id': req_id, 'status': r.status_code,
                               'msg': r.json().get('error', {}).get('message', '')[:60]})
    except Exception as e:
        with lock:
            errors.append({'id': req_id, 'status': 'exception', 'msg': str(e)[:60]})

# Send 50 concurrent requests
N = 50
threads = [threading.Thread(target=hammer_request, args=(i,)) for i in range(N)]
for t in threads: t.start()
for t in threads: t.join()

print(f'\n{N} concurrent requests:')
print(f'  Success: {len(success)}')
print(f'  Errors:  {len(errors)}')
if errors:
    for e in errors[:3]:
        print(f'    {e["status"]}: {e["msg"]}')

In [ ]:
# Cell 4: Check for connection exhaustion logs
wait_for_logs(10)
print('Searching for connection pool errors...')
hits = search_db_errors(minutes_ago=2)
print_logs(hits)

# Also check blocked thread (connection wait)
hits_blocked = search_blocked_thread_errors(minutes_ago=2)
if hits_blocked:
    print('\nBlocked thread warnings (connection wait):')
    print_logs(hits_blocked)

In [ ]:
# Cell 5: Check current pool config vs PostgreSQL capacity
print('=== Connection pool analysis ===')

# Polaris pool config
result = subprocess.run([
    'kubectl', 'exec', '-n', 'datahub-hynix',
    'deployment/benchmarks-polaris',
    '--', 'env'
], capture_output=True, text=True)
pool_vars = [l for l in result.stdout.split('\n') if 'JDBC' in l and 'SIZE' in l]
for v in pool_vars:
    print(f'  {v}')

# PostgreSQL max_connections
max_conn = pg_query('SHOW max_connections;')
print(f'\nPostgreSQL max_connections: {max_conn.strip()}')

print('\n=== Recommendation ===')
print('Rule: pool_size <= (max_connections - reserved) / num_polaris_pods')
print('Example:')
print('  PostgreSQL max_connections: 200')
print('  Reserved for admin/repmgr:  20')
print('  Available:                  180')
print('  Polaris pods (HPA max):     3')
print('  Pool size per pod:          180/3 = 60 (not 300!)')
print('')
print('Your current risk:')
print('  QUARKUS_DATASOURCE_JDBC_MAX_SIZE=300')
print('  But PostgreSQL likely has lower max_connections')
print('  Fix: reduce to 60-100 per pod')

In [ ]:
# Cell 6: Fix recommendation — values.yaml changes
print('Fix 1: Reduce pool size in values.yaml')
print()
print('extraEnv:')
print('  # Reduce max pool to safe level')
print('  - name: QUARKUS_DATASOURCE_JDBC_MAX_SIZE')
print('    value: "60"   # was 300')
print()
print('  # Keep min connections reasonable')
print('  - name: QUARKUS_DATASOURCE_JDBC_MIN_SIZE')
print('    value: "5"    # was 10')
print()
print('  # Fail fast when pool exhausted')
print('  - name: QUARKUS_DATASOURCE_JDBC_ACQUISITION_TIMEOUT')
print('    value: "PT5S"')
print()
print('Fix 2: Increase PostgreSQL max_connections')
print()
print('  kubectl exec -n datahub-hynix \\')
print('    benchmarks-postgresql-postgresql-ha-postgresql-0 --')
print('    psql -U postgres -c')
print('    "ALTER SYSTEM SET max_connections=500;"')
print()
print('Fix 3: Add PgBouncer (production recommended)')
print('  → Connection pooling at PostgreSQL level')
print('  → Polaris connects to PgBouncer, not PostgreSQL directly')
print('  → PgBouncer multiplexes: 300 Polaris → 50 PostgreSQL')

cleanup()
print('\n=== Summary ===')
print('Error:    500 Unable to acquire connection')
print('Cause:    pool_size > PostgreSQL max_connections')
print('Detect:   search_db_errors() + search_blocked_thread_errors()')
print('Fix:      Reduce pool max-size OR increase PostgreSQL max_connections')
print('Formula:  pool_size = (pg_max_conn - reserved) / num_pods')
print('Best:     Add PgBouncer for production workloads')